# Balance Scale Classification

A balance scale tips left, tips right, or stays balanced depending on the weight and distance on each side. The physics is a simple product — left weight times left distance versus right weight times right distance — but can a generic classifier learn that product rule from 625 examples?

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Balance Scale (id 12). Note: the class is the FIRST column here.
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/12/data.csv')))
print(df.shape)
print('Scales per outcome (L = tips left, B = balanced, R = tips right):')
print(df['class'].value_counts())
df.head(3)


(625, 5)
Scales per outcome (L = tips left, B = balanced, R = tips right):
class
R    288
L    288
B     49
Name: count, dtype: int64


,class,left-weight,left-distance,right-weight,right-distance
0,B,1,1,1,1
1,R,1,1,1,2
2,R,1,1,1,3


In [2]:

feats = ['left-weight', 'left-distance', 'right-weight', 'right-distance']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.8089
Logistic Regression accuracy: 0.8662
              precision    recall  f1-score   support

           B       0.00      0.00      0.00        12
           L       0.90      0.89      0.90        73
           R       0.85      0.86      0.86        72

    accuracy                           0.81       157
   macro avg       0.58      0.58      0.58       157
weighted avg       0.81      0.81      0.81       157



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which side of the scale matters most?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


right-distance    0.269939
right-weight      0.254071
left-weight       0.241192
left-distance     0.234798
dtype: float64


Winner: Logistic Regression
